# 面向图像生成与编辑用例的图像评估

## 简介

图像模型正越来越多地用于真实的产品工作流：设计稿、营销素材、虚拟试穿，以及对现有品牌物料的高精度编辑。要在生产环境中信任这些系统，仅仅问“看起来好不好？”是不够的。你需要**可重复、针对具体工作流的评估**，来衡量输出是否满足要求、失败时是否安全，以及能否随时间可预期地改进。

视觉评估（vision evals）比文本评估更难，因为“答案”是一张图像，其中混合了：

- **硬性约束**：精确的文字、数量、属性、局部性（“只修改这个区域”）。  
- **感知质量**：清晰度、连贯性、真实感、审美匹配度。  
- **隐蔽的失败模式**：细微的变形、意外的编辑，或乍看没问题却违反产品要求的小文字错误，在编辑场景中尤其如此。

好的视觉评估**不是**给“一张漂亮的图片”打分，而是评判模型**在特定工作流中是否可靠**。许多视觉效果很出色的图像仍然会失败，原因可能是文字错误、风格不符合品牌，或者编辑超出了预期区域。图像评估衡量的是真实提示词下的**质量、可控性和可用性**，而不仅仅是视觉吸引力。

#### 本指南涵盖的内容

本 cookbook 重点介绍如何针对以下四大类别构建实用的图像评估系统：

**1）图像生成评估**

- 指令遵循（约束是否满足）  
- 文字渲染（准确性、可读性、位置）  
- 风格控制（审美匹配、品牌/角色一致性）  
- 偏好对齐（评分标准标签 + 成对比较）

**2）图像编辑评估**

- 变换正确性（所请求的修改被精确完成）  
- 局部性（编辑只发生在预期位置）  
- 保留性（未要求修改的区域保持不变）  
- 空间控制（编辑应用于正确的实例/区域）

**3）人类反馈对齐**

- 基于评分标准的标签和成对偏好，用于捕捉主观质量和“感觉”  
- 校准技术，让人类判断随时间保持一致

**4）构建评估的策略**

- 从不可妥协的正确性关卡开始  
- 在失败得到控制后，再添加分级的质量指标  
- 为失败模式打标签，以驱动有针对性的迭代

## 构建视觉评估框架

视觉评估框架（harness）是一个小型、可重复的系统，它将“这张图像行不行？”转化为跨提示词、模型和设置的**结构化、可比较的结果**。

从宏观上看，视觉评估遵循与任何 LLM 评估系统相同的循环：

**输入 → 模型 → 输出 → 评分器 → 得分 → 反馈 → 改进**

为了让它能在本 cookbook 的其余部分（生成 + 编辑）中复用，请围绕**三个插件**构建该框架：

1.  **测试用例：** 要运行什么（提示词 + 标准 + 可选的输入图像/遮罩）  
2.  **运行器：** 如何调用模型并保存输出图像  
3.  **评分器：** 如何为输出打分（评分标准、LLM 评审，之后还可以加入人工标签）

**设置**

运行一次即可。它会：
- 创建 API 客户端
- 在文件夹中创建 `images/`。 

In [ ]:
import os
from openai import OpenAI

client = OpenAI()

os.makedirs("../../images", exist_ok=True)

下面是一个精简的 `vision_harness/` 包，你可以直接放入自己的仓库。

如需一个可以扩展的完整可运行框架（可编辑、复现并添加你自己的用例），请参阅 [examples/evals/imagegen_evals](https://github.com/openai/openai-cookbook/tree/main/examples/evals/imagegen_evals)。


### vision_harness/types.py

保持核心类型的通用性，这样你就可以在图像生成*和*图像编辑中复用它们。

In [14]:
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Literal, Optional

TaskType = Literal["image_generation", "image_editing"]
ScoreValue = bool | int | float | str


@dataclass(frozen=True)
class ImageInputs:
    """Editing inputs: one or more reference images + optional mask."""
    image_paths: list[Path]
    mask_path: Optional[Path] = None


@dataclass(frozen=True)
class TestCase:
    """A single evaluable example."""
    id: str
    task_type: TaskType
    prompt: str
    criteria: str
    image_inputs: Optional[ImageInputs] = None


@dataclass(frozen=True)
class ModelRun:
    """One model configuration to evaluate (useful for sweeps)."""
    label: str
    task_type: TaskType
    params: dict[str, Any]  # e.g. {"model": "...", "quality": "...", ...}


@dataclass(frozen=True)
class Artifact:
    """A saved artifact from a run (usually an image)."""
    kind: Literal["image"]
    path: Path
    mime: str = "image/png"


@dataclass
class ModelResponse:
    """Normalized output from any runner."""
    artifacts: list[Artifact] = field(default_factory=list)
    raw: dict[str, Any] = field(default_factory=dict)  # optional debug payload


@dataclass(frozen=True)
class Score:
    key: str
    value: ScoreValue
    reason: str = ""
    tags: Optional[list[str]] = None


### vision_harness/io.py

你会在评分器（LLM 评审）中用到它，有时也会在模型调用中用到。


In [15]:
from __future__ import annotations

import base64
from pathlib import Path

_MIME_BY_SUFFIX = {
    ".jpg": "image/jpeg",
    ".jpeg": "image/jpeg",
    ".png": "image/png",
    ".webp": "image/webp",
}

def image_to_data_url(path: Path) -> str:
    mime = _MIME_BY_SUFFIX.get(path.suffix.lower(), "image/png")
    b64 = base64.b64encode(path.read_bytes()).decode("utf-8")
    return f"data:{mime};base64,{b64}"


### vision_harness/storage.py


In [63]:
from __future__ import annotations

import time
from dataclasses import dataclass
from pathlib import Path

@dataclass
class OutputStore:
    """
    Simple artifact store that writes directly to the root folder.
    Note: This implementation writes directly to `root` (no per-test subfolders),
    which keeps site-relative image paths stable for cookbook rendering.
    """
    root: Path

    def run_dir(self, test_id: str, model_label: str) -> Path:
        # Ignore test/model subfolders; write everything to the root.
        self.root.mkdir(parents=True, exist_ok=True)
        return self.root

    def new_basename(self, prefix: str) -> str:
        created_ms = int(time.time() * 1000)
        return f"{prefix}_{created_ms}"

    def save_png(self, run_dir: Path, basename: str, idx: int, png_bytes: bytes) -> Path:
        out = run_dir / f"{basename}_{idx}.png"
        out.write_bytes(png_bytes)
        return out


### vision_harness/sweeps.py


In [17]:
from __future__ import annotations

from itertools import product
from typing import Any


def grid_sweep(
    *,
    base_label: str,
    task_type: TaskType,
    fixed: dict[str, Any],
    grid: dict[str, list[Any]],
) -> list[ModelRun]:
    keys = list(grid.keys())
    runs: list[ModelRun] = []

    for values in product(*[grid[k] for k in keys]):
        params = dict(fixed)
        label_parts = [base_label]
        for k, v in zip(keys, values):
            params[k] = v
            label_parts.append(f"{k}={v}")
        runs.append(ModelRun(label=",".join(label_parts), task_type=task_type, params=params))

    return runs


### vision_harness/runners.py

两个运行器：一个用于**生成**，一个用于**编辑**。两者都返回一个包含已保存输出图像的标准化 `ModelResponse`。


In [ ]:
from __future__ import annotations

import base64
from contextlib import ExitStack
from typing import Optional

from openai import OpenAI


def _mime_for_path(path) -> str:
    suffix = str(path).lower()
    if suffix.endswith(".png"):
        return "image/png"
    if suffix.endswith(".jpg") or suffix.endswith(".jpeg"):
        return "image/jpeg"
    if suffix.endswith(".webp"):
        return "image/webp"
    return "application/octet-stream"


def _extract_b64_items(images_response) -> list[str]:
    b64_items: list[str] = []
    for item in getattr(images_response, "data", []) or []:
        b64 = getattr(item, "b64_json", None)
        if b64:
            b64_items.append(b64)
    return b64_items


class ImageGenerationRunner:
    """Text-to-image runner."""

    def __init__(self, client: Optional[OpenAI] = None):
        self.client = client or OpenAI()

    def run(self, case: TestCase, run_cfg: ModelRun, store: OutputStore) -> ModelResponse:
        assert case.task_type == "image_generation"
        assert run_cfg.task_type == "image_generation"

        params = dict(run_cfg.params)
        model = params.pop("model")
        n = int(params.pop("n", 1))

        run_dir = store.run_dir(case.id, run_cfg.label)
        basename = store.new_basename(f"gen_{case.id}_{run_cfg.label}")

        images_response = self.client.images.generate(
            model=model,
            prompt=case.prompt,
            n=n,
            **params,
        )

        artifacts: list[Artifact] = []
        for idx, b64_json in enumerate(_extract_b64_items(images_response)):
            png_bytes = base64.b64decode(b64_json)
            out_path = store.save_png(run_dir, basename, idx, png_bytes)
            artifacts.append(Artifact(kind="image", path=out_path))

        return ModelResponse(artifacts=artifacts, raw={"model": model, "params": run_cfg.params})


class ImageEditRunner:
    """Image editing runner (reference image(s) + optional mask)."""

    def __init__(self, client: Optional[OpenAI] = None):
        self.client = client or OpenAI()

    def run(self, case: TestCase, run_cfg: ModelRun, store: OutputStore) -> ModelResponse:
        assert case.task_type == "image_editing"
        assert run_cfg.task_type == "image_editing"
        assert case.image_inputs is not None and case.image_inputs.image_paths

        params = dict(run_cfg.params)
        model = params.pop("model")
        n = int(params.pop("n", 1))

        run_dir = store.run_dir(case.id, run_cfg.label)
        basename = store.new_basename(f"edit_{case.id}_{run_cfg.label}")

        with ExitStack() as stack:
            image_files = []
            for p in case.image_inputs.image_paths:
                f = stack.enter_context(p.open("rb"))
                image_files.append((p.name, f, _mime_for_path(p)))

            mask_file = None
            if case.image_inputs.mask_path:
                mf = stack.enter_context(case.image_inputs.mask_path.open("rb"))
                mask_file = (case.image_inputs.mask_path.name, mf, _mime_for_path(case.image_inputs.mask_path))

            edit_kwargs = dict(
                model=model,
                prompt=case.prompt,
                image=image_files,
                n=n,
                **params,
            )
            if mask_file is not None:
                edit_kwargs["mask"] = mask_file

            images_response = self.client.images.edit(**edit_kwargs)

        artifacts: list[Artifact] = []
        for idx, b64_json in enumerate(_extract_b64_items(images_response)):
            png_bytes = base64.b64decode(b64_json)
            out_path = store.save_png(run_dir, basename, idx, png_bytes)
            artifacts.append(Artifact(kind="image", path=out_path))

        return ModelResponse(artifacts=artifacts, raw={"model": model, "params": run_cfg.params})


### vision_harness/graders.py

一个简洁的评分器接口 + 一个可复用的 **LLM 评审**评分器，只需改变构建评审输入的方式，即可同时用于生成和编辑。

In [19]:
from __future__ import annotations

import json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable, Optional, Protocol

from openai import OpenAI


class Grader(Protocol):
    key: str

    def grade(self, response: ModelResponse, case: TestCase) -> Score | list[Score]: ...


def pick_first_image(response: ModelResponse) -> Optional[Path]:
    for artifact in response.artifacts:
        if artifact.kind == "image":
            return artifact.path
    return None


def build_generation_judge_content(case: TestCase, output_image: Path) -> list[dict]:
    return [
        {
            "type": "input_text",
            "text": f"Prompt:\n{case.prompt}\n\nCriteria:\n{case.criteria}",
        },
        {
            "type": "input_image",
            "image_url": image_to_data_url(output_image),
        },
    ]


def build_editing_judge_content(case: TestCase, output_image: Path) -> list[dict]:
    assert case.image_inputs is not None and case.image_inputs.image_paths
    content: list[dict] = [
        {
            "type": "input_text",
            "text": f"Edit instruction:\n{case.prompt}\n\nCriteria:\n{case.criteria}",
        }
    ]
    for image_path in case.image_inputs.image_paths:
        content.append(
            {
                "type": "input_image",
                "image_url": image_to_data_url(image_path),
            }
        )
    if case.image_inputs.mask_path:
        content.append(
            {
                "type": "input_image",
                "image_url": image_to_data_url(case.image_inputs.mask_path),
            }
        )
    content.append(
        {
            "type": "input_image",
            "image_url": image_to_data_url(output_image),
        }
    )
    return content


def default_schema() -> dict:
    return {
        "type": "object",
        "properties": {
            "pass": {"type": "boolean"},
            "reason": {"type": "string"},
        },
        "required": ["pass", "reason"],
        "additionalProperties": False,
    }


@dataclass
class LLMajRubricGrader:
    """
    Reusable vision LLM-as-judge grader.
    - Provide a system prompt + a content_builder for generation/editing.
    - Optionally provide a custom JSON schema and parser.
    """

    key: str
    system_prompt: str
    content_builder: Callable[[TestCase, Path], list[dict]]
    judge_model: str = "gpt-5.2"
    client: Optional[OpenAI] = None

    json_schema_name: str = "vision_eval_result"
    json_schema: dict = field(default_factory=default_schema)
    result_parser: Optional[Callable[[dict, str], Score | list[Score]]] = None

    def _parse_result(self, data: dict) -> Score | list[Score]:
        if self.result_parser:
            return self.result_parser(data, self.key)
        return Score(
            key=self.key,
            value=bool(data.get("pass", False)),
            reason=(data.get("reason") or "").strip(),
            tags=data.get("tags") or None,
        )

    def grade(self, response: ModelResponse, case: TestCase) -> Score | list[Score]:
        output_image = pick_first_image(response)
        if not output_image:
            return Score(key=self.key, value=False, reason="No output image artifact found")

        client = self.client or OpenAI()
        content = self.content_builder(case, output_image)

        completion = client.responses.create(
            model=self.judge_model,
            input=[
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": content},
            ],
            text={
                "format": {
                    "type": "json_schema",
                    "name": self.json_schema_name,
                    "schema": self.json_schema,
                    "strict": True,
                }
            },
        )

        data = json.loads(completion.output_text)
        return self._parse_result(data)


### vision_harness/evaluate.py

一个简单的评估循环，返回普通的 Python 数据（没有“eval row”类）。后面的章节可以在此基础上编写各自的报告/CSV 工具。


In [20]:
from __future__ import annotations

from typing import Any


def _as_score_list(score_or_scores: Score | list[Score]) -> list[Score]:
    if isinstance(score_or_scores, list):
        return score_or_scores
    return [score_or_scores]


def evaluate(
    *,
    cases: list[TestCase],
    model_runs: list[ModelRun],
    graders: list[Grader],
    output_store: OutputStore,
) -> list[dict[str, Any]]:
    gen_runner = ImageGenerationRunner()
    edit_runner = ImageEditRunner()

    results: list[dict[str, Any]] = []

    for case in cases:
        for run_cfg in model_runs:
            if run_cfg.task_type != case.task_type:
                continue

            if case.task_type == "image_generation":
                response = gen_runner.run(case, run_cfg, output_store)
            elif case.task_type == "image_editing":
                response = edit_runner.run(case, run_cfg, output_store)
            else:
                raise ValueError(f"Unknown task_type: {case.task_type}")

            score_map: dict[str, Any] = {}
            reason_map: dict[str, str] = {}

            for grader in graders:
                scored = grader.grade(response, case)
                for score in _as_score_list(scored):
                    score_map[score.key] = score.value
                    reason_map[score.key] = score.reason

            results.append(
                {
                    "test_id": case.id,
                    "model_label": run_cfg.label,
                    "task_type": case.task_type,
                    "artifact_paths": [str(a.path) for a in response.artifacts],
                    "scores": score_map,
                    "reasons": reason_map,
                    "run_params": run_cfg.params,
                }
            )

    return results


### 结果表格渲染辅助函数

In [29]:
import os
from html import escape
from IPython.display import HTML, display


def summarize_scores(scores: dict[str, object]) -> str:
    return os.linesep.join(f"{k}: {scores[k]}" for k in sorted(scores.keys()))


def summarize_reasons(reasons: dict[str, str]) -> str:
    verdict_reason = (reasons.get("verdict") or "").strip()
    if verdict_reason:
        return verdict_reason
    return os.linesep.join(f"{k}: {reasons[k]}" for k in sorted(reasons.keys()) if reasons[k])


def _pre(text: str) -> str:
    return (
        "<pre style='white-space:pre-wrap; word-break:break-word; margin:0'>"
        f"{escape(text)}"
        "</pre>"
    )


def render_result_table(
    *,
    case: TestCase,
    result: dict[str, object],
    title: str,
) -> None:
    sep = os.linesep
    prompt_text = f"{case.prompt}{sep}{sep}Criteria:{sep}{case.criteria}"
    scores = result["scores"]
    reasons = result["reasons"]

    prompt_html = _pre(prompt_text)
    scores_html = _pre(summarize_scores(scores))
    reasons_html = _pre(summarize_reasons(reasons))

    table_html = (
        "<table style='width:100%; table-layout:fixed; border-collapse:collapse;'>"
        "<colgroup>"
        "<col style='width:33%'>"
        "<col style='width:33%'>"
        "<col style='width:33%'>"
        "</colgroup>"
        "<thead><tr>"
        "<th style='text-align:left; padding:8px; border-bottom:1px solid #ddd'>Input Prompt</th>"
        "<th style='text-align:left; padding:8px; border-bottom:1px solid #ddd'>Scores</th>"
        "<th style='text-align:left; padding:8px; border-bottom:1px solid #ddd'>Reasoning</th>"
        "</tr></thead>"
        "<tbody><tr>"
        f"<td style='text-align:left; padding:8px; vertical-align:top'>{prompt_html}</td>"
        f"<td style='text-align:left; padding:8px; vertical-align:top'>{scores_html}</td>"
        f"<td style='text-align:left; padding:8px; vertical-align:top'>{reasons_html}</td>"
        "</tr></tbody></table>"
    )

    display(HTML(f"<div style='font-weight:600; margin:6px 0'>{escape(title)}</div>"))
    display(HTML(table_html))


后续的 cookbook 章节只需提供：

1.  该用例的 `TestCase` 对象列表（UI 设计稿、营销图形、虚拟试穿、Logo 编辑等）  
2.  一组 `ModelRun`（单次运行，或使用 `grid_sweep` 进行参数扫描）  
3.  一组评分器，通常包括：
    - **关卡型评分器**（通过/失败）：不可妥协的要求，例如指令遵循、文字正确性、局部性/保留性  
    - **分级指标**（0–5）：质量维度，例如真实感/可用性、布局/层级、瑕疵严重程度  
    - 可选：人工评分标准标签和成对偏好（存储在框架之外）

这样，cookbook 的其余部分就可以专注于每个用例**要衡量什么**，而不必每次都重写框架的底层管道。

## 图像生成评估

**用例思路：UI 设计稿、营销图形/海报**

**目标：评估真实提示词下文生图的质量、可控性和实用性**  
涵盖：

- **指令遵循**（约束是否满足），  
- **文字渲染**（生成的文字准确、清晰且位置正确），  
- **风格**（所要求的审美 + 视觉质量，可能包括品牌、角色和产品一致性）  
- **人类反馈对齐**（基于评分标准的标签 + 成对偏好）。

图像生成模型被用来创作影响实际工作的产物：UI 设计稿、产品设计、营销图形和内部演示。在这些场景中，图像不仅仅以视觉吸引力来评判，还要看它们是否传达了意图、遵循了约束，以及能否被下游相关人员使用。

### UI 设计稿

由图像模型生成的 UI 设计稿正越来越多地用于早期产品探索、设计构思和内部评审。在这些工作流中，设计稿不只是视觉灵感，它们还是沟通工具，帮助设计师、工程师和相关人员在动手构建之前，对布局、层级、交互意图和功能范围进行推敲。因此，成功的标准与其说是审美品味，不如说是输出能否可信地充当产品产物。

示例：

<img src="../../images/mockup_sample.png" width="400"/> 

本节通过一个屏幕级示例展示如何评估生成的 UI 设计稿。目标是评估生成的 UI 是否具有连贯的布局、使用了可识别的组件，并呈现出一个可信的产品体验。

**为什么 UI 设计稿评估与众不同**

UI 设计稿结合了几个难以评估的维度：

1.  **组件保真度（关卡）**  
    生成的图像必须清楚地呈现所要求的屏幕类型和状态。按钮应该看起来可点击，输入框应该看起来可编辑，导航应该看起来像导航。  
2.  **布局实现（分级）**  
    好的 UI 设计稿能让人一眼看出用户可以做什么、什么最重要。布局应让主要操作一目了然，次要操作明显处于从属地位，信息的分组方式应反映真实的交互流程  
3.  **图内文字渲染（关卡）**  
    UI 文字承载着功能。标签、标题和行动号召（CTA）必须清晰可读且渲染正确。

**示例任务：生成移动端结账页面**

- 场景：一个面向消费者的电商应用需要一个移动端结账页面，用于核对订单并完成付款。
- 提示词指导：明确说明屏幕类型、平台、必需的 UI 元素、精确的按钮/链接文字、层级约束以及禁止出现的多余内容。
- 实现：完整的提示词和标准位于下方的框架设置代码单元格中（`ui_prompt` 和 `ui_criteria`）。

#### 评估什么（实用指标）：

##### 1）指令遵循：通过 / 失败（关卡）

大多数 UI 设计稿评估都从指令保真度开始。从根本上说，问题很简单：模型生成的是不是所要求的屏幕？如果提示词指定的是移动端结账页面，输出就应该看起来像移动端结账页面，而不是一个通用的落地页。必需的部分、状态或约束都应该存在，整体结构应与请求的意图相符。这一维度通常权重最高，因为如果模型没能满足核心要求，输出的其余部分就无关紧要了。一旦 UI 设计稿满足了基本指令，评估重点就转向该屏幕作为产品产物是否可用、是否连贯。

**通过条件**

- 存在正确的屏幕类型和平台上下文。  
- 包含所有必需的部分、状态或约束。

**失败条件**

- 缺少任何必需组件，或输出改变了 UI 的用途。

##### 2）布局与层级：0–5

衡量屏幕能否让人一眼看懂。

**关注要点：**

- 主要操作与次要操作之间有清晰的视觉层级  
- 对齐和间距保持一致  
- 相关元素按逻辑分组

##### 3）文字渲染与可读性：通过 / 失败（关卡）

标签、标题和行动号召需要清晰可读、没有歧义。

**通过条件**

- 文字可读、拼写正确，标签合理。  
- 字号体现层级（标题、标签与辅助文字之间的区分）。

**失败条件**

- 任何关键文字无法阅读、被截断、拼写错误或变形。

##### 4）UI 真实感与可用性：*0–5*

衡量设计稿是否像一个可信的产品界面。

**关注要点：**

- 输入框看起来可编辑 / 按钮看起来可点击  
- 导航看起来像导航  
- 交互元素在视觉上与静态内容区分开

##### 5）人类反馈（快速且高杠杆）

UI 的“可用性清晰度”是人类最能发挥价值的地方。保持轻量，并聚焦于交互意图。

- 评分标准标签（快速）：
  - 你会用这个设计稿来迭代真实的产品页面吗？（是/否）
  - 如果否，原因是什么？（缺少部分、操作按钮不清晰、文字无法阅读、布局混乱等）
  - 整体可用性清晰度：1–5
- 常见失败标签（用于调试 + 迭代）：
  - `wrong_screen_type`
  - `missing_required_section`
  - `extra_ui_elements`
  - `primary_cta_not_clear`
  - `text_unreadable_or_garbled`
  - `affordances_unclear`
  - `layout_confusing`

**判定规则（如何将各项指标转换为单一的通过/失败）**

- **指令遵循必须通过**  
- **布局与层级 ≥ 3**  
- **文字渲染必须通过**  
- **UI 真实感 ≥ 3**

任何一条规则不满足 → 整体**失败**。

**`TestCase` 集合示例（规模小但信号强）**

先从几个覆盖常见 UI 设计稿变体和边缘情况的用例开始。

1.  **移动端** **结账页面**（即上文的“结账 UI 设计稿”提示词）  
2.  **极简布局**（仅包含页头 + 订单总额 + 主要 CTA）：在元素稀少的情况下测试布局和层级。  
3.  **密集层级**（订单摘要 + 支付方式 + 优惠码行 + 税费 + 两个次要操作）：测试信息负载下的层级。  
4.  **精确的 CTA 文字**（主按钮文字必须恰好为“Place Order”）：测试图内文字渲染的保真度。  
5.  **次要操作的存在**（同时包含“Edit Cart”和“Change payment method”）：测试次要操作在视觉上是否保持从属地位。  
6.  **位置约束**（要求订单总额紧挨在主要 CTA 的正上方）  
7.  **平台保真度**（移动端与桌面端版本）：测试屏幕框架和平台特征。

**LLM 评审评分标准提示词**

下面是一个与现有 `LLMajRubricGrader` 对齐的评审提示词。它会返回结构化的指标得分 + 一个总体判定。

你可以通过修改 JSON schema 以包含以下字段，将其与现有评分器一起使用（如果愿意，也可以为每个指标分别创建评分器）。

#### 系统提示词（UI 设计稿评审）


In [ ]:
PROMPT = """<core_mission>
Evaluate whether a generated UI mockup image represents a usable mobile checkout screen by checking screen type fidelity, layout/hierarchy, in-image text rendering, and UI affordance clarity.
</core_mission>

<role>
You are an expert evaluator of UI mockups used by designers and engineers. You care about structural correctness, readable UI text, clear hierarchy, and realistic rendering of UI elements.
</role>

<scope_constraints>
- Judge only against the provided instructions.
- Be strict about required UI elements and exact button/link text.
- Do NOT infer intent beyond what is explicitly stated.
- Do NOT reward creativity that violates constraints.
- Missing or extra required components are serious errors.
- If the UI intent or function is unclear, score conservatively.
</scope_constraints>

<metrics>
1) instruction_following: PASS/FAIL
2) layout_hierarchy: 0–5
3) in_image_text_rendering: PASS/FAIL
4) ui_affordance_rendering: 0–5
</metrics>

Evaluate EACH metric independently using the definitions below.
--------------------------------
1) Instruction Following (PASS / FAIL)
--------------------------------
PASS if:
- All required components are present.
- No unrequested components or features are added.
- The screen matches the requested type and product context.

FAIL if:
- Any required component is missing.
- Any unrequested component materially alters the UI.
- The screen does not match the requested type.

--------------------------------
2) Layout and Hierarchy (0–5)
--------------------------------
5: Layout is clear, coherent, and immediately usable.
   Hierarchy, grouping, spacing, and alignment are strong.

3: Generally understandable, but one notable hierarchy or layout issue
   that would require iteration.

0-2: Layout problems materially hinder usability or comprehension.

--------------------------------
3) In-Image Text Rendering (PASS / FAIL)
--------------------------------
PASS if:
- Text is readable, correctly spelled, and sensibly labeled.
- Font sizes reflect hierarchy (headings vs labels vs helper text).

FAIL if:
- Any critical text is unreadable, cut off, misspelled, or distorted.

--------------------------------
4) UI Affordance Rendering (0–5)
--------------------------------
5: Clearly resembles a real product interface that designers could use.

3: Marginally plausible; intent is visible but execution is weak.

0-2: Poor realism; interface would be difficult to use in practice.

<verdict_rules>
- Instruction Following must PASS.
- In-Image Text Rendering must PASS.
- Layout and Hierarchy score must be ≥ 3.
- UI Affordance Rendering score must be ≥ 3.

If ANY rule fails, the overall verdict is FAIL.
Do not average scores to determine the verdict.
</verdict_rules>

<output_constraints>
Return JSON only.
No extra text.
</output_constraints>
"""

#### 推荐的 JSON 输出


In [25]:
{
  "verdict": "PASS",
  "instruction_following": True,
  "layout_hierarchy": 3,
  "in_image_text_rendering": True,
  "ui_affordance_rendering": 4,
  "reason": "..."
}


{'verdict': 'PASS',
 'instruction_following': True,
 'layout_hierarchy': 3,
 'in_image_text_rendering': True,
 'ui_affordance_rendering': 4,
 'reason': '...'}



### UI 设计稿示例：框架设置

定义一个 UI 设计稿测试用例、一次模型运行，以及位于 `images/` 下的输出存储。


In [31]:
# Capture the UI judge prompt before PROMPT is overwritten later.
ui_judge_prompt = PROMPT

ui_prompt = """Generate a high-fidelity mobile checkout screen for an ecommerce app.
Orientation: portrait.
Screen type: checkout / order review.
Use the REQUIRED TEXT:
- Checkout
- Place Order
- Edit Cart
Constraints:
- Order total appears directly above the primary CTA.
- Primary CTA is the most visually prominent element.
- Do not include popups, ads, marketing copy, or extra screens.
- Do not include placeholder or lorem ipsum text.
"""

ui_criteria = """The image clearly depicts a mobile checkout screen.
All required sections are present and visually distinct.
UI elements look clickable/editable and follow common conventions.
Primary vs secondary actions are unambiguous.
No extra UI states, decorative noise, or placeholder text."""

ui_case = TestCase(
    id="ui_checkout_mockup",
    task_type="image_generation",
    prompt=ui_prompt,
    criteria=ui_criteria,
)

ui_run = ModelRun(
    label="gpt-image-1.5-ui",
    task_type="image_generation",
    params={
        "model": "gpt-image-1.5",
        "n": 1,
        "size": "1024x1024",
    },
)

ui_store = OutputStore(root=Path("../../images"))




### UI 设计稿示例：运行与评分

运行框架，并使用 UI 评审评分标准为 UI 设计稿评分。


In [34]:
ui_schema = {
    "type": "object",
    "properties": {
        "verdict": {"type": "string"},
        "instruction_following": {"type": "boolean"},
        "layout_hierarchy": {"type": "number"},
        "in_image_text_rendering": {"type": "boolean"},
        "ui_affordance_rendering": {"type": "number"},
        "reason": {"type": "string"},
    },
    "required": [
        "verdict",
        "instruction_following",
        "layout_hierarchy",
        "in_image_text_rendering",
        "ui_affordance_rendering",
        "reason",
    ],
    "additionalProperties": False,
}


def parse_ui_result(data: dict, base_key: str) -> list[Score]:
    return [
        Score(key="instruction_following", value=bool(data["instruction_following"]), reason=""),
        Score(key="layout_hierarchy", value=float(data["layout_hierarchy"]), reason=""),
        Score(key="in_image_text_rendering", value=bool(data["in_image_text_rendering"]), reason=""),
        Score(key="ui_affordance_rendering", value=float(data["ui_affordance_rendering"]), reason=""),
        Score(key="verdict", value=str(data["verdict"]), reason=(data.get("reason") or "").strip()),
    ]

ui_grader = LLMajRubricGrader(
    key="ui_eval",
    system_prompt=ui_judge_prompt,
    content_builder=build_generation_judge_content,
    judge_model="gpt-5.2",
    json_schema_name="ui_mockup_eval",
    json_schema=ui_schema,
    result_parser=parse_ui_result,
)

ui_results = evaluate(
    cases=[ui_case],
    model_runs=[ui_run],
    graders=[ui_grader],
    output_store=ui_store,
)

ui_result = ui_results[0]
ui_result


{'test_id': 'ui_checkout_mockup',
 'model_label': 'gpt-image-1.5-ui',
 'task_type': 'image_generation',
 'artifact_paths': ['../../images/gen_ui_checkout_mockup_gpt-image-1.5-ui_1769656414221_0.png'],
 'scores': {'instruction_following': True,
  'layout_hierarchy': 5.0,
  'in_image_text_rendering': True,
  'ui_affordance_rendering': 5.0,
  'verdict': 'PASS'},
 'reasons': {'instruction_following': '',
  'layout_hierarchy': '',
  'in_image_text_rendering': '',
  'ui_affordance_rendering': '',
  'verdict': 'All required text is present ("Checkout", "Edit Cart", "Place Order"). Screen clearly matches a mobile checkout/order review with distinct sections (Shipping Information, Payment Method, Items, Order Summary). Order Total appears directly above the primary CTA, and the "Place Order" button is the most visually prominent element. No popups, ads, marketing copy, or lorem/placeholder text. Text is crisp and readable with appropriate hierarchy. UI elements (back arrow, Edit Cart link, Chan



### UI 设计稿结果

在一个 pandas 表格中展示提示词、生成的图像和得分。


<img src="../../images/gen_ui_checkout_mockup_gpt-image-1.5-ui_1769656414221_0.png" width="400"/> 

In [35]:
render_result_table(case=ui_case, result=ui_result, title="UI Mockup: Prompt vs. Scores")


Input Prompt,Scores,Reasoning
"Generate a high-fidelity mobile checkout screen for an ecommerce app. Orientation: portrait. Screen type: checkout / order review. Use the REQUIRED TEXT: - Checkout - Place Order - Edit Cart Constraints: - Order total appears directly above the primary CTA. - Primary CTA is the most visually prominent element. - Do not include popups, ads, marketing copy, or extra screens. - Do not include placeholder or lorem ipsum text. Criteria: The image clearly depicts a mobile checkout screen. All required sections are present and visually distinct. UI elements look clickable/editable and follow common conventions. Primary vs secondary actions are unambiguous. No extra UI states, decorative noise, or placeholder text.",in_image_text_rendering: True instruction_following: True layout_hierarchy: 5.0 ui_affordance_rendering: 5.0 verdict: PASS,"All required text is present (""Checkout"", ""Edit Cart"", ""Place Order""). Screen clearly matches a mobile checkout/order review with distinct sections (Shipping Information, Payment Method, Items, Order Summary). Order Total appears directly above the primary CTA, and the ""Place Order"" button is the most visually prominent element. No popups, ads, marketing copy, or lorem/placeholder text. Text is crisp and readable with appropriate hierarchy. UI elements (back arrow, Edit Cart link, Change links, primary CTA) have clear, realistic affordances."


### 营销图形生成

营销图形是一种“高风险的文生图”工作流：输出是要正式发布的（或者至少要按照可以发布的标准来审阅）。一张传单可能看起来“很好看”，但如果文案错误、优惠信息不清楚，或者布局掩盖了关键信息，它仍然是失败的。

本节以一家咖啡店为例，展示如何评估**传单生成**。目标是让评估与工作流相关：营销人员或设计师能否只需少量修改就使用这个输出？

示例：

<img src="../../images/sample_flyer.png" width="400"/> 

**为什么营销图形评估与众不同**

营销图像结合了三类难题：

1.  精确的文字要求（关卡）
2.  布局 + 层级（分级）
3.  风格 + 品牌一致性（分级 + 可选的人工偏好）

**示例任务：生成咖啡店传单**

- 场景：一家本地咖啡店需要一张宣传传单，推广一款限时饮品和一项工作日优惠。
- 提示词指导：明确说明交付物、必需的文案、层级以及禁止出现的多余内容。
- 实现：完整的提示词和标准位于下方的框架设置代码单元格中（`coffee_generation_prompt` 和 `coffee_criteria`）。

#### 评估什么（实用指标）：

结合使用关卡（硬性约束）和分级信号（质量）。

##### 1）指令遵循：通过 / 失败（关卡）

如果营销素材不符合规格，就会直接失败。

**通过条件**

- 交付物类型正确：明显是传单/海报布局（不是 UI 页面，也不是随意的照片）。
- 必需组件齐全（主标题/副标题/优惠/CTA/页脚 + 主视觉图）。

**失败条件**

- 缺少任何必需组件，或输出无法被识别为传单。

##### 2）文字渲染准确性：通过 / 失败（关卡）

这通常是生产环境中排名第一的失败模式。

**通过条件**

- 所有必需的文字串都存在且完全正确（拼写、标点、大小写、符号）。
- 文字可读（没有涂抹、截断、扭曲或重叠）。

**失败条件**

- 任何必需文字错误/缺失/无法阅读，或出现了任何多余文字。

提示：如果你的工作流要求“文案完全一致”，就把这一项当作硬性关卡。不要通过取平均把它稀释掉。

##### 3）布局与层级：0–5

衡量传单能否让人一眼看懂。

**关注要点：**

- 优先级清晰：主标题占主导，副标题起辅助作用，优惠信息突出，页脚处于次要位置。
- 对齐和间距显得有意为之（没有拥挤的元素堆，没有随意漂浮的元素）。
- 阅读顺序明确无歧义。

##### 4）风格与品牌契合度：0–5

衡量是否符合所要求的“氛围”，并避免与品牌不符的外观。

**关注要点：**

- 温暖、舒适、简约；精品咖啡的视觉语言。
- 配色和字体给人的感觉保持一致（没有多种相互冲突的风格）。
- 如果提示词要求照片级写实，则应避免卡通化的插画。

##### 5）视觉质量与瑕疵严重程度：0–5

一张传单即使“正确”，如果视觉上有缺陷，仍然无法使用。

**关注要点：**

- 没有变形的物体、破损的手/杯子、融化的奶泡纹理，文字周围也没有奇怪的瑕疵。
- 背景纹理低调（不与文案争夺注意力）。
- 主视觉图看起来诱人且连贯。

##### 6）人类反馈（快速且高杠杆）

UI 的“可用性清晰度”是人类最能发挥价值的地方。保持轻量，并聚焦于交互意图。

- 评分标准标签（快速）：
  - 你会用这个设计稿来迭代真实的产品页面吗？（是/否）
  - 如果否，原因是什么？（缺少部分、操作按钮不清晰、文字无法阅读、布局混乱等）
  - 整体可用性清晰度：1–5
- 常见失败标签（用于调试 + 迭代）：
  - `wrong_screen_type`
  - `missing_required_section`
  - `extra_ui_elements`
  - `primary_cta_not_clear`
  - `text_unreadable_or_garbled`
  - `affordances_unclear`
  - `layout_confusing`

**判定规则（如何将各项指标转换为单一的通过/失败）**

一套简单而严格的规则效果很好：

- **指令遵循必须通过**
- **文字渲染必须通过**
- **布局与层级 ≥ 3**
- **风格与品牌契合度 ≥ 3**
- **视觉质量 ≥ 3**

任何一条规则不满足 → 整体**失败**。  
（不要用平均分来抵消文字正确性的问题。）

**`TestCase` 集合示例（规模小但信号强）**

先用大约 8–12 个用例覆盖常见的传单变体和边缘情况。

1. 季节性活动（即上文的“Winter Latte Week”提示词）
2. 极少文字（只有 2 行 + Logo + 营业时间）：测试稀疏布局
3. 信息密集（菜单亮点 + 优惠 + 两个 CTA）：测试信息负载下的层级
4. 严格禁止多余文字：测试凭空生成的填充文案
5. 精确的标点（“20% OFF • Mon–Thu”）：测试符号保真度
6. 两项优惠（“BOGO Tuesdays” + “Happy Hour 2–4”）：测试多徽章布局
7. 配色约束（“仅使用奶油色 + 深棕色 + 柔和橙色”）
8. 无障碍变体（高对比度、大字号、简单背景）

**LLM 评审评分标准提示词**

下面是一个与现有 `LLMajRubricGrader` 对齐的评审提示词。它会返回结构化的指标得分 + 一个总体判定。

你可以通过修改 JSON schema 以包含以下字段，将其与现有评分器一起使用（如果愿意，也可以为每个指标分别创建评分器）。


In [38]:
PROMPT = """<core_mission>
Evaluate whether a generated marketing flyer is usable for a real coffee shop
promotion by checking instruction adherence, exact text correctness, layout clarity,
style fit, and artifact severity.
</core_mission>

<role>
You are an expert evaluator of marketing design deliverables.
You care about correctness, readability, hierarchy, and brand-fit.
You do NOT reward creativity that violates constraints.
</role>

<scope_constraints>
- Judge ONLY against the provided prompt and criteria.
- Be strict about required copy: spelling, punctuation, casing, and symbols must match exactly.
- Extra or missing text is a serious error.
- If unsure, score conservatively (lower score).
</scope_constraints>

<metrics>
1) instruction_following: PASS/FAIL
2) text_rendering: PASS/FAIL
3) layout_hierarchy: 0-5
4) style_brand_fit: 0-5
5) visual_quality: 0-5

Use these anchors:
Layout/Hierarchy 5 = instantly readable; clear order; strong spacing/alignment.
3 = understandable but needs iteration (one clear issue).
0-2 = confusing or hard to parse.

Style/Brand Fit 5 = clearly matches requested vibe; consistent; not off-style.
3 = generally matches but with noticeable mismatch.
0-2 = wrong style (e.g. cartoonish when photo-real requested).

Visual Quality 5 = clean; no distracting artifacts; hero image coherent.
3 = minor artifacts but still usable.
0-2 = obvious artifacts or distortions that break usability.
</metrics>

<verdict_rules>
Overall verdict is FAIL if:
- instruction_following is FAIL, OR
- text_rendering is FAIL, OR
- any of layout_hierarchy/style_brand_fit/visual_quality is < 3.
Otherwise PASS.
</verdict_rules>

<output_constraints>
Return JSON only.
No extra text.
</output_constraints>
"""


#### 推荐的 JSON 输出


In [39]:
{
  "verdict": "PASS",
  "instruction_following": True,
  "text_rendering": True,
  "layout_hierarchy": 4,
  "style_brand_fit": 4,
  "visual_quality": 3,
  "reason": "..."
}

{'verdict': 'PASS',
 'instruction_following': True,
 'text_rendering': True,
 'layout_hierarchy': 4,
 'style_brand_fit': 4,
 'visual_quality': 3,
 'reason': '...'}



### 咖啡示例：框架设置

定义一个营销传单测试用例、一次模型运行，以及一个输出存储。


In [42]:
# Run the harness for a single coffee flyer generation case.

coffee_generation_prompt = """Create a print-ready vertical A4 flyer for a coffee shop called Sunrise Coffee.
Use a warm, cozy, minimal specialty coffee aesthetic (not cartoonish).
Required text (must be exact):
- WINTER LATTE WEEK
- Try our Cinnamon Oat Latte
- 20% OFF - Mon-Thu
- Order Ahead
- 123 Market St - 7am-6pm
Do not include any other words, prices, URLs, or QR codes.
"""

coffee_criteria = """All required text appears exactly as written and is legible.
Layout reads clearly: shop name -> headline -> subheadline -> offer -> CTA -> footer.
Style matches warm, cozy, specialty coffee and is not cartoonish.
No extra text, watermarks, or irrelevant UI-like elements."""

coffee_case = TestCase(
    id="coffee_flyer_generation",
    task_type="image_generation",
    prompt=coffee_generation_prompt,
    criteria=coffee_criteria,
)

coffee_run = ModelRun(
    label="gpt-image-1.5",
    task_type="image_generation",
    params={
        "model": "gpt-image-1.5",
        "n": 1,
        "size": "1024x1024",
    },
)

# Save artifacts under the repo images/ folder so they render on the site.
coffee_store = OutputStore(root=Path("../../images"))


### 咖啡示例：运行与评分

运行框架，并使用营销评审评分标准为输出打分。


In [48]:
# Use the prebuilt graders + evaluate helpers with a structured marketing schema.
from openai import OpenAI

marketing_schema = {
    "type": "object",
    "properties": {
        "verdict": {"type": "string"},
        "instruction_following": {"type": "boolean"},
        "text_rendering": {"type": "boolean"},
        "layout_hierarchy": {"type": "number"},
        "style_brand_fit": {"type": "number"},
        "visual_quality": {"type": "number"},
        "reason": {"type": "string"},
    },
    "required": [
        "verdict",
        "instruction_following",
        "text_rendering",
        "layout_hierarchy",
        "style_brand_fit",
        "visual_quality",
        "reason",
    ],
    "additionalProperties": False,
}


def parse_marketing_result(data: dict, base_key: str) -> list[Score]:
    return [
        Score(key="instruction_following", value=bool(data["instruction_following"]), reason=""),
        Score(key="text_rendering", value=bool(data["text_rendering"]), reason=""),
        Score(key="layout_hierarchy", value=float(data["layout_hierarchy"]), reason=""),
        Score(key="style_brand_fit", value=float(data["style_brand_fit"]), reason=""),
        Score(key="visual_quality", value=float(data["visual_quality"]), reason=""),
        Score(key="verdict", value=str(data["verdict"]), reason=(data.get("reason") or "").strip()),
    ]


marketing_grader = LLMajRubricGrader(
    key="marketing_eval",
    system_prompt=PROMPT,
    content_builder=build_generation_judge_content,
    judge_model="gpt-5.2",
    json_schema_name="marketing_flyer_eval",
    json_schema=marketing_schema,
    result_parser=parse_marketing_result,
)

coffee_results = evaluate(
    cases=[coffee_case],
    model_runs=[coffee_run],
    graders=[marketing_grader],
    output_store=coffee_store,
)

coffee_result = coffee_results[0]

# Simple alternative: exact text-rendering check via OCR-style extraction.
REQUIRED_TEXT = {
    "WINTER LATTE WEEK",
    "Try our Cinnamon Oat Latte",
    "20% OFF • Mon–Thu",
    "Order Ahead",
    "123 Market St • 7am–6pm",
}


def extract_text_from_flyer(image_path: str | Path, model: str = "gpt-5.2") -> list[str]:
    judge_client = client if "client" in globals() else OpenAI()
    image_url = image_to_data_url(Path(image_path))

    instructions = (
        "List every piece of text visible in this flyer image. "
        "Return one line per text item and preserve capitalization, punctuation, and spacing exactly."
    )

    response = judge_client.responses.create(
        model=model,
        instructions=instructions,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_image", "image_url": image_url},
                ],
            }
        ],
    )

    return [line.strip() for line in (response.output_text or "").splitlines() if line.strip()]


coffee_image_path = Path(coffee_result["artifact_paths"][0])
extracted_lines = extract_text_from_flyer(coffee_image_path)

missing = sorted(REQUIRED_TEXT - set(extracted_lines))
extra = sorted(set(extracted_lines) - REQUIRED_TEXT)
text_rendering_pass = not missing and not extra

parts = []
if missing:
    parts.append("missing: " + ", ".join(missing))
if extra:
    parts.append("extra: " + ", ".join(extra))
text_rendering_reason = "; ".join(parts) if parts else "Exact text match."

# Keep both: the judge's text_rendering and the OCR exact-match result.
text_rendering_judge_score = coffee_result["scores"].get("text_rendering")
text_rendering_judge_reason = coffee_result["reasons"].get("text_rendering", "")

coffee_result["scores"]["text_rendering_judge"] = text_rendering_judge_score
coffee_result["reasons"]["text_rendering_judge"] = text_rendering_judge_reason

coffee_result["scores"]["text_rendering_ocr"] = text_rendering_pass
coffee_result["reasons"]["text_rendering_ocr"] = text_rendering_reason

coffee_result


{'test_id': 'coffee_flyer_generation',
 'model_label': 'gpt-image-1.5',
 'task_type': 'image_generation',
 'artifact_paths': ['../../images/gen_coffee_flyer_generation_gpt-image-1.5_1769657762450_0.png'],
 'scores': {'instruction_following': True,
  'text_rendering': True,
  'layout_hierarchy': 5.0,
  'style_brand_fit': 5.0,
  'visual_quality': 5.0,
  'verdict': 'PASS',
  'text_rendering_judge': True,
  'text_rendering_ocr': False},
 'reasons': {'instruction_following': '',
  'text_rendering': '',
  'layout_hierarchy': '',
  'style_brand_fit': '',
  'visual_quality': '',
  'verdict': 'All required lines appear and match exactly (including casing, hyphen/en-dash, and punctuation), with no extra words, prices, URLs, or QR codes. Text is crisp and legible. Hierarchy follows the requested order: shop name at top, then headline, subheadline, offer, CTA, and footer address/hours. Visual style is warm, cozy, minimal specialty coffee with a realistic latte photo (not cartoonish). No visible wa



### 传单生成结果

并排渲染生成的图像、提示词和得分。


<img src="../../images/gen_coffee_flyer_generation_gpt-image-1.5_1769657762450_0.png" width="400"/> 

In [49]:
render_result_table(case=coffee_case, result=coffee_result, title="Coffee Flyer: Prompt vs. Scores")


Input Prompt,Scores,Reasoning
"Create a print-ready vertical A4 flyer for a coffee shop called Sunrise Coffee. Use a warm, cozy, minimal specialty coffee aesthetic (not cartoonish). Required text (must be exact): - WINTER LATTE WEEK - Try our Cinnamon Oat Latte - 20% OFF - Mon-Thu - Order Ahead - 123 Market St - 7am-6pm Do not include any other words, prices, URLs, or QR codes. Criteria: All required text appears exactly as written and is legible. Layout reads clearly: shop name -> headline -> subheadline -> offer -> CTA -> footer. Style matches warm, cozy, specialty coffee and is not cartoonish. No extra text, watermarks, or irrelevant UI-like elements.",instruction_following: True layout_hierarchy: 5.0 style_brand_fit: 5.0 text_rendering: True text_rendering_judge: True text_rendering_ocr: False verdict: PASS visual_quality: 5.0,"All required lines appear and match exactly (including casing, hyphen/en-dash, and punctuation), with no extra words, prices, URLs, or QR codes. Text is crisp and legible. Hierarchy follows the requested order: shop name at top, then headline, subheadline, offer, CTA, and footer address/hours. Visual style is warm, cozy, minimal specialty coffee with a realistic latte photo (not cartoonish). No visible watermarks, UI elements, or distracting artifacts; overall print-flyer quality is clean."


#### 补充方法：图像转文本 + 文本比较

另一种评估方法是将问题转换到另一种模态，也就是文本。在这种设置中，先使用视觉模型将图像转换为详细的文字描述，然后通过将生成的文本与预期文本进行比较来完成评估。

这种方法特别适合 OCR 类工作流。例如，你可以使用视觉模型从生成的传单中提取文字，然后将得到的文本行直接与所需的文案集合进行比较，以验证其正确性和完整性。

In [ ]:
import pandas as pd

coffee_image_path = Path(coffee_result["artifact_paths"][0])
extracted_lines = extract_text_from_flyer(coffee_image_path)

required_lines = sorted(REQUIRED_TEXT)
required_set = set(required_lines)
extracted_set = set(extracted_lines)

missing = sorted(required_set - extracted_set)
extra = sorted(extracted_set - required_set)

text_rendering_df = pd.DataFrame(
    {
        "text_rendering_score": [coffee_result["scores"].get("text_rendering")],
        "missing_text": ["\n".join(missing) or "(none)"],
        "extra_text": ["\n".join(extra) or "(none)"],
        "extracted_text": ["\n".join(extracted_lines) or "(no text extracted)"],
    }
)

text_rendering_df


,text_rendering_score,missing_text,extra_text,extracted_text
0,True,123 Market St • 7am–6pm\n20% OFF • Mon–Thu,123 Market St - 7am-6pm\n123 Market St - 7am-6...,SUNRISE COFFEE\nWINTER LATTE WEEK\nTry our Cin...


## 图像编辑评估


### 虚拟试穿

虚拟试穿（VTO）是一种**图像编辑**工作流：给定一张**人物照片**（自拍或模特照）和一个**服装参考**（商品照片和/或描述），生成一张服装看起来**自然穿在身上**的输出图，同时保持人物的身份、姿势和场景不变。

**为什么 VTO 评估与众不同**

与“创意类”编辑不同，VTO 以**保真度 + 保留性**来评判：

- **保留穿着者**（面部身份、体型、姿势）  
- **保留商品**（颜色、图案、Logo、材质特征）  
- **只编辑需要编辑的部分**（局部性/保留性）  
- **看起来符合物理规律**（遮挡、光照、垂坠感、褶皱）

示例输入：

| 全身照 | 单品 1 |
|:------------:|:--------------:|
| ![](../../images/woman_in_museum.png)  | ![](../../images/jacket.png) |
| 单品 2 | 单品 3 |
| ![](../../images/tank_top.png) | ![](../../images/boots.png) |

输出图像： 

<img src="../../images/outfit.png" width="400"/> 

#### 评估什么（实用指标）：

##### 1）将 VTO 视为多参考图编辑任务

对于每个测试用例，存储：

- **输入人物图像**（自拍/模特照）  
- **商品参考**（商品目录图、平铺图、人台图，或在可用时提供“上身”参考图）  
- 可选但高杠杆的内容：
  - **遮罩**：可编辑区域、服装区域、头发/手部遮挡物  
  - **元数据**：类别（上装/下装/外套）、期望的版型（宽松/修身）、配色、长度  
  - **编辑指令**：“穿上*这件一模一样的夹克*，不要改变背景或面部。”

这样评分器就可以将**输出与两个输入**（人物 + 商品）进行比较，而不只是问“看起来好不好？”

##### 2）分级指标（只使用以下三项）

将这些作为 **0–5** 分的得分，用于对模型排名并跟踪改进。让它们保持**相互独立**（不要在评分器内部取平均）；如果需要关卡，请在外部使用判定规则。

##### A）面部相似度（输出 vs 自拍）：0–5

衡量输出是否保留了*同一个人*（身份），而不仅仅是“一张看起来合理的脸”。

- **5**：明显是同一个人；关键面部特征不变；没有明显的年龄/种族/风格漂移；表情变化（如有）细微且真实。  
- **4**：同一个人；只有仔细观察才能发现的微小差异（轻微的形状/纹理平滑，眼睛/嘴巴略有偏移）。  
- **3**：大体是同一个人，但至少存在一处明显的身份漂移（五官比例、下颌线、眼睛、鼻子），会降低用户信任。  
- **2**：显著的身份漂移；看起来像另一个人或面部被大幅修改。  
- **1**：严重损坏（面部融化/模糊）或明显是另一个身份。  
- **0**：面部缺失、无法辨认或被替换。

关注要点：

- 面部几何的一致性（眼睛/鼻子/嘴巴间距、下颌线、颧骨）  
- 皮肤纹理真实，不会因“美颜滤镜”而丢失身份特征  
- 除非有要求，否则不出现意外的妆容/年龄/风格变化

##### B）服装保真度（输出 vs 提供的单品）：0–5

衡量输出中的服装是否与用户所选的*具体*商品参考相符。

- **5**：单品与参考高度一致：类别、配色、图案/印花、材质特征和关键细节（Logo、接缝、领口、口袋）都正确。  
- **4**：明显是同一件单品；有 1–2 处细微偏差（Logo 略微模糊、色调轻微偏移、细节略有简化）。  
- **3**：总体正确，但存在明显不符（图案比例错误、材质看起来不同、关键设计元素缺失/多出）。  
- **2**：多处不符；可能是另一个款式变体或另一件商品。  
- **1**：单品类别错误，或视觉特征严重不符。  
- **0**：服装未被应用 / 缺失 / 被替换为无关的衣物。

关注要点：

- 颜色/图案的正确性（尤其是条纹、格纹、小型重复图案）  
- Logo/文字的完整性（没有凭空生成的字母）  
- 结构细节（领口、袖子、下摆、扣合方式）

##### C）体型保留（输出 vs 自拍）：0–5

衡量模型是否保留了穿着者的体型、姿势和比例，**正常服装效果之外的部分**不应改变（例如，宽松的衣服可以改变轮廓，但不应重塑身体结构）。

- **5**：身体比例和姿势得以保留；服装自然贴合，没有扭曲躯干/四肢。  
- **4**：有与服装相符的细微、合理的轮廓变化；没有明显的身体结构变形。  
- **3**：明显的重塑（腰/臀/肩/四肢），让人觉得有点“AI 风格化”，或与输入的身体不一致。  
- **2**：严重扭曲（四肢被拉长、关节移位、躯干被压缩），在产品使用中不可接受。  
- **1**：严重的身体结构变形（多出/缺少肢体，身体部位融化）。  
- **0**：身体无法辨认，或从根本上被损坏。

关注要点：

- 肩宽/臀宽与输入相比保持一致  
- 四肢长度/关节位置稳定（肘部、膝盖、手腕）  
- 没有“瘦身”或“膨胀”的瑕疵

如果你需要单一的总体通过/失败结果，常见的规则是：

- 任何指标 ≤ 2 则**失败**  
- 所有指标 ≥ 3 则**通过**  
  （对于商业关键流程，还可以额外要求服装保真度 ≥ 4）。

##### 3）人类反馈：保持简单但一致

人类最擅长判断“我会在购物流程中信任这个结果吗？”

使用两种标签类型：

- **评分标准标签**（快速、结构化）：
  - 身份是否得以保留？（是/否）  
  - 服装是否与参考相符？（是/否 + 问题所在）  
  - 是否有不良瑕疵？（无/轻微/严重）  
  - 总体上是否可用于电商？（是/否）  
- **成对偏好**（A vs B）：
  - 哪个输出在保持人物不变的同时，对商品更忠实？

定期进行**校准**：保留一小组“锚点”示例，让评分人员重新打分，以防止标准漂移。



### 虚拟试穿示例：框架设置

使用 `images/` 中已有的图像作为人物和服装参考。


In [51]:
vto_person_path = Path("../../images/base_woman.png")
vto_garment_path = Path("../../images/jacket.png")

vto_prompt = """Put the person in the first image into the jacket shown in the second image.
Keep the person's face, pose, body shape, and background unchanged.
Preserve the garment's color, pattern, and key details.
Do not add extra accessories, text, or new elements."""

vto_criteria = """The output preserves the same person and background.
The jacket matches the reference garment closely.
Body shape and pose remain consistent outside normal garment effects.
The result looks physically plausible."""

vto_case = TestCase(
    id="vto_jacket_tryon",
    task_type="image_editing",
    prompt=vto_prompt,
    criteria=vto_criteria,
    image_inputs=ImageInputs(image_paths=[vto_person_path, vto_garment_path]),
)

vto_run = ModelRun(
    label="gpt-image-1.5-vto",
    task_type="image_editing",
    params={
        "model": "gpt-image-1.5",
        "n": 1,
    },
)

vto_store = OutputStore(root=Path("../../images"))




### 虚拟试穿示例：运行与评分

定义一个与 VTO 指标对齐的 VTO 评审提示词，并运行框架。


In [55]:
vto_judge_prompt = """<core_mission>
Evaluate whether a virtual try-on edit preserves the person while accurately applying the reference garment.
</core_mission>

<role>
You are an expert evaluator of virtual try-on outputs.
You focus on identity preservation, garment fidelity, and body-shape preservation.
</role>

<metrics>
1) facial_similarity: 0-5
2) outfit_fidelity: 0-5
3) body_shape_preservation: 0-5
</metrics>

<verdict_rules>
FAIL if any metric <= 2.
PASS if all metrics >= 3.
</verdict_rules>

<output_constraints>
Return JSON only with the fields specified in the schema.
</output_constraints>
"""

vto_schema = {
    "type": "object",
    "properties": {
        "verdict": {"type": "string"},
        "facial_similarity": {"type": "number"},
        "outfit_fidelity": {"type": "number"},
        "body_shape_preservation": {"type": "number"},
        "reason": {"type": "string"},
    },
    "required": [
        "verdict",
        "facial_similarity",
        "outfit_fidelity",
        "body_shape_preservation",
        "reason",
    ],
    "additionalProperties": False,
}


def parse_vto_result(data: dict, base_key: str) -> list[Score]:
    return [
        Score(key="facial_similarity", value=float(data["facial_similarity"]), reason=""),
        Score(key="outfit_fidelity", value=float(data["outfit_fidelity"]), reason=""),
        Score(key="body_shape_preservation", value=float(data["body_shape_preservation"]), reason=""),
        Score(key="verdict", value=str(data["verdict"]), reason=(data.get("reason") or "").strip()),
    ]

vto_grader = LLMajRubricGrader(
    key="vto_eval",
    system_prompt=vto_judge_prompt,
    content_builder=build_editing_judge_content,
    judge_model="gpt-5.2",
    json_schema_name="vto_eval",
    json_schema=vto_schema,
    result_parser=parse_vto_result,
)

vto_results = evaluate(
    cases=[vto_case],
    model_runs=[vto_run],
    graders=[vto_grader],
    output_store=vto_store,
)

vto_result = vto_results[0]
vto_result


{'test_id': 'vto_jacket_tryon',
 'model_label': 'gpt-image-1.5-vto',
 'task_type': 'image_editing',
 'artifact_paths': ['../../images/edit_vto_jacket_tryon_gpt-image-1.5-vto_1769658750053_0.png'],
 'scores': {'facial_similarity': 5.0,
  'outfit_fidelity': 4.0,
  'body_shape_preservation': 4.0,
  'verdict': 'PASS'},
 'reasons': {'facial_similarity': '',
  'outfit_fidelity': '',
  'body_shape_preservation': '',
  'verdict': 'The edited output preserves the same face, hairstyle, pose, and plain studio background. The applied jacket closely matches the reference: camel/beige color, notch lapels, single-breasted look with dark buttons, and flap pockets are present and placed plausibly. Minor deviations include slightly different button count/placement and subtle differences in lapel/hem shaping compared to the flat lay. Body proportions and stance remain consistent, with only natural silhouette changes from wearing a structured blazer.'},
 'run_params': {'model': 'gpt-image-1.5', 'n': 1}}

### 虚拟试穿：可选的 Code Interpreter 裁剪工具

如果你需要更细粒度的证据（Logo、接缝、版型、面部细节），可以运行一个辅助评审，使用 Code Interpreter 的 `crop` 工具放大特定区域。
这对于近距离检查服装保真度和身份保留很有用。

In [54]:
vto_output_path = Path(vto_result["artifact_paths"][0])

instructions = """
Tools available:
- crop(image_id, x1, y1, x2, y2): Use to zoom into a specific image's region. Coordinates are integer pixels relative to the top-left of the CURRENT view of that image. Use as few precise crops as necessary to gather evidence.
    - When using crop(image_id, x1, y1, x2, y2), ensure that x2 > x1 and y2 > y1. The coordinates must define a valid rectangle: x1 (left, inclusive), y1 (top, inclusive), x2 (right, exclusive), y2 (bottom, exclusive), with x2 strictly greater than x1 and y2 strictly greater than y1. If you are unsure, double-check your coordinates before cropping to avoid errors.

Images provided:
- Up to 5 user reference photos (ref_1..ref_5) for identity/baseline body context
- 1 clothing-only image (clothing)
- 1 reconstruction image (recon)

Your goals:
1) Judge similarity between the clothing-only image and how it appears on the user in the reconstruction.
2) Judge identity realism (face/hair/skin) vs. user reference photos.
3) Judge overall realism (lighting, shadows, artifacts).

IMPORTANT:
- Use the crop tool when you need more detail. After crop, a new grid overlay is returned for that image.
- You may use the crop tool as many times as needed to gather evidence.
- When confident, produce the final STRICT JSON only. No extra text.
"""
judge_client = client if "client" in globals() else OpenAI()
response_crop = judge_client.responses.create(
    model="gpt-5.2",
    instructions=instructions,
    tools=[
        {
            "type": "code_interpreter",
            "container": {"type": "auto", "memory_limit": "4g"},
        }
    ],
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": "Order: ref_1 (person), clothing (garment reference), recon (try-on output).",
                },
                {
                    "type": "input_image",
                    "image_url": image_to_data_url(vto_person_path),
                },
                {
                    "type": "input_image",
                    "image_url": image_to_data_url(vto_garment_path),
                },
                {
                    "type": "input_image",
                    "image_url": image_to_data_url(vto_output_path),
                },
            ],
        }
    ],
)

response_crop

Response(id='resp_03756a1c45c8427000697ad91445ec8196a58b39ee7e0b05b1', created_at=1769658644.0, error=None, incomplete_details=None, instructions="\nTools available:\n- crop(image_id, x1, y1, x2, y2): Use to zoom into a specific image's region. Coordinates are integer pixels relative to the top-left of the CURRENT view of that image. Use as few precise crops as necessary to gather evidence.\n    - When using crop(image_id, x1, y1, x2, y2), ensure that x2 > x1 and y2 > y1. The coordinates must define a valid rectangle: x1 (left, inclusive), y1 (top, inclusive), x2 (right, exclusive), y2 (bottom, exclusive), with x2 strictly greater than x1 and y2 strictly greater than y1. If you are unsure, double-check your coordinates before cropping to avoid errors.\n\nImages provided:\n- Up to 5 user reference photos (ref_1..ref_5) for identity/baseline body context\n- 1 clothing-only image (clothing)\n- 1 reconstruction image (recon)\n\nYour goals:\n1) Judge similarity between the clothing-only ima



### 虚拟试穿评估结果

在一个 pandas 表格中展示编辑结果和 VTO 得分。


| 全身照 | 单品 1 |
|:------------:|:--------------:|
| ![](../../images/base_woman.png)  | ![](../../images/jacket.png) |

编辑后的图像：

<img src="../../images/edit_vto_jacket_tryon_gpt-image-1.5-vto_1769658750053_0.png" width="400"/> 


In [57]:
render_result_table(case=vto_case, result=vto_result, title="Virtual Try-On: Prompt vs. Scores")


Input Prompt,Scores,Reasoning
"Put the person in the first image into the jacket shown in the second image. Keep the person's face, pose, body shape, and background unchanged. Preserve the garment's color, pattern, and key details. Do not add extra accessories, text, or new elements. Criteria: The output preserves the same person and background. The jacket matches the reference garment closely. Body shape and pose remain consistent outside normal garment effects. The result looks physically plausible.",body_shape_preservation: 4.0 facial_similarity: 5.0 outfit_fidelity: 4.0 verdict: PASS,"The edited output preserves the same face, hairstyle, pose, and plain studio background. The applied jacket closely matches the reference: camel/beige color, notch lapels, single-breasted look with dark buttons, and flap pockets are present and placed plausibly. Minor deviations include slightly different button count/placement and subtle differences in lapel/hem shaping compared to the flat lay. Body proportions and stance remain consistent, with only natural silhouette changes from wearing a structured blazer."


### Logo 编辑

Logo 编辑是一项高精度的图像编辑任务。给定一个**现有 Logo** 和一条**范围严格限定的指令**，模型必须精确地应用所要求的修改，同时完美保留其他所有内容。与创意设计任务不同，Logo 编辑通常只有一个正确答案。任何偏差，哪怕再细微，都算失败。


<img src="../../images/logo_sample.png" width="400"/> 


**为什么 Logo 编辑评估与众不同**：

Logo 编辑以精确性、局部性和保留性来评判，而不是视觉吸引力：

- 保留原始素材的识别特征  
- 保留所有未编辑的文字、几何形状、间距和样式  
- 只编辑明确要求的区域  
- 达到字符级的正确性，对漂移零容忍

小错误会带来不成比例的风险。哪怕一个字母变形、一个数字改变，或编辑溢出预期区域，都可能破坏品牌完整性，并造成下游返工。

**示例任务：**

- 输入：Logo 图像 + 遮罩或区域描述  
- 任务：“将年份从 2024 改为 2026”“将 C 替换为 S”“添加 TM”

#### 评估什么（实用指标）：

##### 1）将 Logo 编辑视为受约束的单参考图任务

对于每个测试用例，存储：

- **输入 Logo 图像**  
- 范围明确的**编辑指令**  
- 可选但高杠杆的内容：
  - 定义允许编辑位置的遮罩或区域描述  
  - 用于精确比较的预期目标文字或符号  
  - **元数据**：字体类型、字号或颜色

评分器应将输出直接与原始 Logo 进行比较。

##### 2）分级指标

使用这些 **0–5** 分的得分对模型排名并跟踪改进。得分针对所有请求的步骤整体给出，而不是逐步给出，这样部分完成的情况会以一种可控且可解释的方式受到惩罚。

##### A）编辑意图正确性：0–5

衡量每个请求的编辑步骤是否都正确地应用到了正确的目标上。

- **5**：所有编辑步骤都完全按要求应用。每一步的字符级准确度都完美无缺。  
- **4**：所有步骤都正确应用；只有仔细观察才能看到极其细微的视觉瑕疵。  
- **3**：所有步骤都已应用，但至少有一步在清晰度或精确度上明显下降。  
- **2**：大多数步骤正确应用，但有一个或多个步骤存在实质性错误。  
- **1**：一个或多个步骤不正确，或应用到了错误的元素上。  
- **0**：大多数步骤缺失、错误或应用不当。

##### B）非目标区域不变性：0–5

衡量在所有步骤中，请求编辑之外的内容是否保持不变。

- **5**：在请求的编辑之外没有可检测到的变化。  
- **4**：只有仔细观察才能看到极其细微的漂移。  
- **3**：附近元素存在明显但有限的漂移。  
- **2**：出现了明显的非请求修改，影响到相邻的文字、符号或背景。  
- **1**：整个 Logo 出现大范围的意外修改。  
- **0**：Logo 的识别特征遭到破坏。

##### C）字符与样式完整性：0–5

Logo 编辑不是创意性的变换。输出必须保留原始素材的识别特征，包括颜色、笔画、字形和图标的一致性。

- **5**：编辑后的字符和符号与原始样式完美匹配。颜色、笔画、字形和图标与原图无法区分。  
- **4**：只有仔细观察才能看到极其细微的偏差，不影响品牌认知。  
- **3**：一项或多项属性存在明显但有限的偏差，但不影响识别。  
- **2**：颜色、笔画、字形或图标几何上存在明显的不一致，影响视觉统一性。  
- **1**：存在重大不一致，实质性地改变了 Logo 的外观。  
- **0**：视觉体系遭到破坏或已无法识别。

**LLM 评审评分标准提示词**

下面是一个与现有 `LLMajRubricGrader` 对齐的评审提示词。它会返回结构化的指标得分 + 一个总体判定。

你可以通过修改 JSON schema 以包含以下字段，将其与现有评分器一起使用（如果愿意，也可以为每个指标分别创建评分器）。


In [58]:
PROMPT = """<core_mission>
Evaluate whether a logo edit was executed with exact correctness,
strict preservation, and high visual integrity.

Logo editing is a precision task.
Small errors matter.
Near-misses are failures.
</core_mission>

<role>
You are an expert evaluator of high-precision logo and brand asset editing.
You specialize in detecting subtle text errors, unintended changes,
and preservation drift across single-step and multi-step edits.
</role>

<scope_constraints>
- Judge only against the provided edit instruction and input logo.
- Do NOT judge aesthetics or visual appeal.
- Do NOT infer intent beyond what is explicitly stated.
- Be strict, conservative, and consistent across cases.
</scope_constraints>

<metrics_and_scoring>

Evaluate EACH metric independently using the definitions below.
All metrics are scored from 0 to 5.
Scores apply across ALL requested edit steps.

--------------------------------
1) Edit Intent Correctness (0–5)
--------------------------------
Measures whether every requested edit step was applied correctly
to the correct target.

5: All edit steps applied exactly as specified. Character-level
   accuracy is perfect for every step.
4: All steps applied correctly with extremely minor visual
   imperfections visible only on close inspection.
3: All steps applied, but one or more steps show noticeable
   degradation in clarity or precision.
2: Most steps applied correctly, but one or more steps contain
   a meaningful error.
1: One or more steps are incorrect or applied to the wrong element.
0: Most steps missing, incorrect, or misapplied.

What to consider:
- Exact character identity (letters, numbers, symbols)
- Correct sequencing and targeting of multi-step edits
- No ambiguous characters (Common confusions: 0 vs 6, O vs D, R vs B)

--------------------------------
2) Non-Target Invariance (0–5)
--------------------------------
Measures whether content outside the requested edits remains unchanged.

5: No detectable changes outside the requested edits.
4: Extremely minor drift visible only on close inspection.
3: Noticeable but limited drift in nearby elements.
2: Clear unrequested changes affecting adjacent text,
   symbols, or background.
1: Widespread unintended changes across the logo.
0: Logo identity compromised.

What to consider:
- Adjacent letter deformation or spacing shifts
- Background, texture, or color changes
- Cumulative drift from multi-step edits

--------------------------------
3) Character and Style Integrity (0–5)
--------------------------------
Measures whether the edited content preserves the original
logo’s visual system.

This includes color, stroke weight, letterform structure,
and icon geometry.

5: Edited characters and symbols perfectly match the original
   style. Colors, strokes, letterforms, and icons are
   indistinguishable from the original.
4: Extremely minor deviation visible only on close inspection,
   with no impact on brand perception.
3: Noticeable but limited deviation in one or more properties
   that does not break recognition.
2: Clear inconsistency in color, stroke, letterform, or icon
   geometry that affects visual cohesion.
1: Major inconsistency that materially alters the logo’s appearance.
0: Visual system is corrupted or no longer recognizable.

</metrics_and_scoring>

<verdict_rules>
- Edit Intent Correctness must be ≥ 4.
- Non-Target Invariance must be ≥ 4.
- Character and Style Integrity must be ≥ 4.

If ANY metric falls below threshold, the overall verdict is FAIL.
Do not average scores to determine the verdict.
</verdict_rules>

<consistency_rules>
- Score conservatively.
- If uncertain between two scores, choose the lower one.
- Base all scores on concrete visual observations.
- Penalize cumulative degradation across multi-step edits.
</consistency_rules>

<output_constraints>
Return JSON only.
No additional text.
</output_constraints>
"""


#### 推荐的 JSON 输出


In [59]:
{
  "verdict": "PASS",
  "edit_intent_correctness": 5,
  "non_target_invariance": 5,
  "character_and_style_integrity": 5,
  "reason": "..."
}


{'verdict': 'PASS',
 'edit_intent_correctness': 5,
 'non_target_invariance': 5,
 'character_and_style_integrity': 5,
 'reason': '...'}



### Logo 编辑示例：框架设置

使用一个现有的 Logo 图像和一条范围严格限定的编辑指令。


In [60]:
# Capture the logo judge prompt.
logo_judge_prompt = PROMPT

logo_input_path = Path("../../images/logo_generation_1.png")

logo_prompt = """Edit the logo by changing the text from FIELD to BUTTER .
Do not change any other text, colors, shapes, or layout."""

logo_criteria = """The requested edit is applied exactly.
All non-target content remains unchanged.
Character style, color, and geometry remain consistent with the original."""

logo_case = TestCase(
    id="logo_year_edit",
    task_type="image_editing",
    prompt=logo_prompt,
    criteria=logo_criteria,
    image_inputs=ImageInputs(image_paths=[logo_input_path]),
)

logo_run = ModelRun(
    label="gpt-image-1.5-logo",
    task_type="image_editing",
    params={
        "model": "gpt-image-1.5",
        "n": 1,
    },
)

logo_store = OutputStore(root=Path("../../images"))




### Logo 编辑示例：运行与评分

运行框架，并使用 Logo 评审评分标准为 Logo 编辑打分。


In [61]:
logo_schema = {
    "type": "object",
    "properties": {
        "verdict": {"type": "string"},
        "edit_intent_correctness": {"type": "number"},
        "non_target_invariance": {"type": "number"},
        "character_and_style_integrity": {"type": "number"},
        "reason": {"type": "string"},
    },
    "required": [
        "verdict",
        "edit_intent_correctness",
        "non_target_invariance",
        "character_and_style_integrity",
        "reason",
    ],
    "additionalProperties": False,
}


def parse_logo_result(data: dict, base_key: str) -> list[Score]:
    return [
        Score(key="edit_intent_correctness", value=float(data["edit_intent_correctness"]), reason=""),
        Score(key="non_target_invariance", value=float(data["non_target_invariance"]), reason=""),
        Score(key="character_and_style_integrity", value=float(data["character_and_style_integrity"]), reason=""),
        Score(key="verdict", value=str(data["verdict"]), reason=(data.get("reason") or "").strip()),
    ]

logo_grader = LLMajRubricGrader(
    key="logo_eval",
    system_prompt=logo_judge_prompt,
    content_builder=build_editing_judge_content,
    judge_model="gpt-5.2",
    json_schema_name="logo_eval",
    json_schema=logo_schema,
    result_parser=parse_logo_result,
)

logo_results = evaluate(
    cases=[logo_case],
    model_runs=[logo_run],
    graders=[logo_grader],
    output_store=logo_store,
)

logo_result = logo_results[0]
logo_result


{'test_id': 'logo_year_edit',
 'model_label': 'gpt-image-1.5-logo',
 'task_type': 'image_editing',
 'artifact_paths': ['../../images/edit_logo_year_edit_gpt-image-1.5-logo_1769659071403_0.png'],
 'scores': {'edit_intent_correctness': 5.0,
  'non_target_invariance': 0.0,
  'character_and_style_integrity': 2.0,
  'verdict': 'FAIL'},
 'reasons': {'edit_intent_correctness': '',
  'non_target_invariance': '',
  'character_and_style_integrity': '',
  'verdict': 'Target text was correctly changed from “FIELD” to “BUTTER” (now reads “BUTTER & FLOUR”). However, major unrequested changes occurred: the background changed from a gray gradient to solid black, and the logo’s overall rendering/contrast differs (the original had a soft glow/embossed look, while the edited version is flatter with different tonal values). These violate the instruction to not change any other colors, shapes, or layout.'},
 'run_params': {'model': 'gpt-image-1.5', 'n': 1}}



### Logo 编辑评估结果

在一个 pandas 表格中展示编辑后的 Logo 和 Logo 编辑得分。


<img src="../../images/edit_logo_year_edit_gpt-image-1.5-logo_1769659071403_0.png" width="400"/> 

In [62]:
render_result_table(case=logo_case, result=logo_result, title="Logo Editing: Prompt vs. Scores")


Input Prompt,Scores,Reasoning
"Edit the logo by changing the text from FIELD to BUTTER . Do not change any other text, colors, shapes, or layout. Criteria: The requested edit is applied exactly. All non-target content remains unchanged. Character style, color, and geometry remain consistent with the original.",character_and_style_integrity: 2.0 edit_intent_correctness: 5.0 non_target_invariance: 0.0 verdict: FAIL,"Target text was correctly changed from “FIELD” to “BUTTER” (now reads “BUTTER & FLOUR”). However, major unrequested changes occurred: the background changed from a gray gradient to solid black, and the logo’s overall rendering/contrast differs (the original had a soft glow/embossed look, while the edited version is flatter with different tonal values). These violate the instruction to not change any other colors, shapes, or layout."


##### 3）已有真实参考图像时的替代方案

如果你已经有标准参考，可以直接与**设计上的权威来源**进行比较，而不是与模型输出比较。

真实参考应来自设计系统，而不是生成的图像：

- 品牌库素材（矢量图或高分辨率的标准位图）
- 表示正确素材状态的 Figma 画框或组件

这与 Logo 编辑的要求能够清晰对应，例如“仍然是同一个素材”以及“意外漂移即为失败”。

**示例：OpenAI Logo – 农历新年**

- 参考图像：标准 OpenAI Logo（来自 Figma 或品牌素材）
- 生成图像：应用了所请求变换的模型输出



## 结论

图像模型正在从“酷炫的演示”转变为生产工具，生成真正的产物：页面、传单、产品视觉图和品牌编辑，这些产物会影响决策并交付给客户。本 cookbook 的核心经验很简单：你无法用通用的“看起来不错”的分数来评估这些系统。你需要**针对具体工作流的评估**，并且这些评估在不同模型版本、提示词和设置之间都**可重复**。多模态 LLM 通过充当可扩展的评审让这一切变得切实可行，前提是配合严格的评分标准、结构化输出和人工校准。

一个实用的视觉评估方案从小处着手，并随着时间推移不断打磨：

- **从关卡开始。** 针对会破坏实际工作的失败模式添加严格的通过/失败评分器：缺少必需组件、文案错误、编辑溢出预期区域，或对应保留区域的意外修改。这可以防止“好看但错误”的输出掩盖回归问题。  
- **逐层加入分级指标。** 一旦硬性失败得到控制，就使用 0–5 分的评分标准，捕捉每个工作流中对可用性和质量真正重要的方面（例如 UI 设计稿中的层级、营销中的品牌契合度，或编辑中的保真度/保留性）。  
- **为失败打标签以加快迭代。** 一致的失败标签能把一堆输出转化为可执行的工程工作：你可以量化哪些地方出了问题，找到集中的根本原因，并在调整提示词、模型设置、遮罩或后处理时跟踪进展。  
- **有策略地使用人工。** 人类在主观或模糊的维度上（“氛围”、可用性清晰度、可信度）最能发挥价值，但前提是保持评分标准严格，并使用校准锚点防止漂移。  
- **把框架当作产品来对待。** 你构建的可复用框架（测试用例、运行器、评分器和存储的产物）为回归测试、参数扫描和并排比较奠定了基础。随着时间推移，你的评估套件会成为安全网：它能及早发现细微的失败，并让改进变得可衡量。
- **按需演进。** 随着时间推移，你可以针对复杂指标重新设计评分器提示词。一个自然的下一步是将单次评分器提示词拆分为多次调用，每次聚焦于一个特定指标。

构建能够反映图像实际使用方式的评估，先确保正确性再追求美观，并让迭代由数据驱动。当你的评估与真实的工作流要求保持一致时，图像生成和编辑就不再是不可预测的艺术项目，而会成为团队可以信赖的工具。